View at:

https://laurauguc.github.io/3d-sound-system/

In [56]:
from dataclasses import dataclass
import numpy as np
import plotly.graph_objects as go


# =========================================================
# CONFIGURATION
# =========================================================

SALA_WIDTH = 4.25
SALA_DEPTH = 4.13

STAGE_WIDTH = 3.70
STAGE_DEPTH = 3.72
STAGE_HEIGHT = .7

# Speakers dimensions
speakers = {'Polk RC55i': {'width': .192 , 'height': .279}, # Overall speaker dimensions (in meters)
           'Polk 255c-RT':  {'width': .376 , 'height': .237}}

# Ceiling disk coordinates
side_shift = 1.65
forward_shift = 1.3

Lp_y = round(SALA_DEPTH / 2 - side_shift,2)
Rp_y = round(SALA_DEPTH / 2 + side_shift,2)
TMp_x = round(SALA_WIDTH - forward_shift,2)

F_side_speaker_width = speakers['Polk RC55i']['width']
inward_shift = -(STAGE_DEPTH - Rp_y - F_side_speaker_width / 2) + .02 # extra 2 cm margin

Rp_y = Rp_y - inward_shift
Lp_y = Lp_y + inward_shift
side_shift = round( side_shift -inward_shift,2)

@dataclass
class BoxObject:
    name: str
    coords: list[tuple[float, float]]
    height: float
    color: str
    base_z: float = 0.0


# =========================================================
# GEOMETRY HELPERS
# =========================================================

def rectangle(x: float, y: float, width: float, depth: float):
    """Return rectangle corner coordinates."""
    return [
        (x, y),
        (x + width, y),
        (x + width, y + depth),
        (x, y + depth),
    ]


def centered_against_wall(
    room_width: float,
    room_depth: float,
    obj_width: float,
    obj_depth: float,
):
    """Place object centered on the right wall."""
    x = room_width - obj_depth
    y = (room_depth - obj_width) / 2
    return x, y


def circle_polygon(
    cx: float,
    cy: float,
    radius: float,
    segments: int = 48,
):
    """Return polygon approximation of a circle."""
    angles = np.linspace(
        0,
        2 * np.pi,
        segments,
        endpoint=False,
    )

    return [
        (
            cx + radius * np.cos(a),
            cy + radius * np.sin(a),
        )
        for a in angles
    ]


# =========================================================
# FURNITURE
# =========================================================

COUCH_WIDTH = 2.40
COUCH_DEPTH = 0.97
COUCH_HEIGHT = 0.90

couch_x, couch_y = centered_against_wall(
    SALA_WIDTH,
    SALA_DEPTH,
    COUCH_WIDTH,
    COUCH_DEPTH,
)

couch_coords = rectangle(
    couch_x,
    couch_y,
    COUCH_DEPTH,
    COUCH_WIDTH,
)


# =========================================================
# SCENE OBJECTS
# =========================================================

objects = [
    BoxObject(
        name="SALA",
        coords=rectangle(
            0,
            0,
            SALA_WIDTH,
            SALA_DEPTH,
        ),
        height=3.3,
        color="skyblue",
    ),
    BoxObject(
        name="STAGE",
        coords=rectangle(
            -STAGE_WIDTH,
            0,
            STAGE_WIDTH,
            STAGE_DEPTH,
        ),
        height=STAGE_HEIGHT,
        color="saddlebrown",
    ),
    BoxObject(
        name="COUCH",
        coords=couch_coords,
        height=COUCH_HEIGHT,
        color="saddlebrown",
    ),

    # Ceiling-mounted disks
    BoxObject(
        name="TML",
        coords=circle_polygon(
            TMp_x,
            Lp_y,
            radius=0.1016,  # 20.32 cm diameter 
        ),
        height=0.001,      # 1 mm thick
        base_z=3.299,      # flush with 3.3 m ceiling
        color="gold",
    ),
    BoxObject(
        name="TMR",
        coords=circle_polygon(
            TMp_x,
            Rp_y,
            radius=0.1016,
        ),
        height=0.001,
        base_z=3.299,
        color="gold",
    ),
]


# =========================================================
# MESH GENERATION
# =========================================================

def extrude_polygon(
    coords,
    height,
    color,
    name,
    base_z=0.0,
    opacity=0.7,
):
    coords = np.asarray(coords)
    n = len(coords)

    bottom = np.column_stack(
        [coords, np.full(n, base_z)]
    )

    top = np.column_stack(
        [coords, np.full(n, base_z + height)]
    )

    vertices = np.vstack([bottom, top])

    x, y, z = vertices.T

    i, j, k = [], [], []
    offset = n

    # Bottom face
    for t in range(1, n - 1):
        i.append(0)
        j.append(t)
        k.append(t + 1)

    # Top face
    for t in range(1, n - 1):
        i.append(offset)
        j.append(offset + t + 1)
        k.append(offset + t)

    # Side walls
    for s in range(n):
        nxt = (s + 1) % n

        i.extend([s, s])
        j.extend([nxt, offset + nxt])
        k.extend([offset + nxt, offset + s])

    return go.Mesh3d(
        x=x,
        y=y,
        z=z,
        i=i,
        j=j,
        k=k,
        color=color,
        opacity=opacity,
        flatshading=True,
        name=name,
    )


# =========================================================
# FIGURE
# =========================================================

fig = go.Figure()

label_x = []
label_y = []
label_z = []
label_text = []

for obj in objects:
    fig.add_trace(
        extrude_polygon(
            obj.coords,
            obj.height,
            obj.color,
            obj.name,
            base_z=obj.base_z,
        )
    )

    center = np.mean(obj.coords, axis=0)

    label_x.append(center[0])
    label_y.append(center[1])
    label_z.append(
        obj.base_z + obj.height + 0.15
    )
    label_text.append(obj.name)


fig.add_trace(
    go.Scatter3d(
        x=label_x,
        y=label_y,
        z=label_z,
        mode="text",
        text=label_text,
        textposition="middle center",
        textfont=dict(
            size=16,
            color="black",
        ),
        showlegend=False,
    )
)

fig.update_layout(
    title="Interactive 3D Polygon Scene",
    width=1100,
    height=850,
    scene=dict(
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)",
        aspectmode="data",
        camera=dict(
            eye=dict(
                x=1.8,
                y=-2.0,
                z=1.4,
            )
        ),
    ),
)

# =========================================================
# CEILING REFERENCE LINES
# =========================================================

distance_from_wall = round(SALA_WIDTH - TMp_x,2)

fig.add_trace(
    go.Scatter3d(
        x=[TMp_x, SALA_WIDTH],
        y=[Lp_y, Lp_y],
        z=[3.3, 3.3],
        mode="lines",
        line=dict(
            color="red",
            width=6,
        ),
        name=f"Distance from wall = {distance_from_wall} m",
    )
)

fig.add_trace(
    go.Scatter3d(
        x=[TMp_x, SALA_WIDTH],
        y=[Rp_y, Rp_y],
        z=[3.3, 3.3],
        mode="lines",
        line=dict(
            color="red",
            width=6,
        ),
        name=f"Distance from wall = {distance_from_wall} m",
        showlegend=False
    )
)

# Line 3
fig.add_trace(
    go.Scatter3d(
        x=[SALA_WIDTH, SALA_WIDTH],
        y=[Lp_y, SALA_DEPTH / 2],
        z=[3.3, 3.3],
        mode="lines",
        line=dict(
            color="green",
            width=6,
        ),
        name=f"Distance from center = {side_shift} m",
    )
)

# Line 4
fig.add_trace(
    go.Scatter3d(
        x=[SALA_WIDTH, SALA_WIDTH],
        y=[Rp_y, SALA_DEPTH / 2],
        z=[3.3, 3.3],
        mode="lines",
        line=dict(
            color="green",
            width=6,
        ),
        name=f"Distance from center = {side_shift} m",
        showlegend=False
    )
)

fig.add_trace(
    go.Scatter3d(
        x=[
            TMp_x, SALA_WIDTH,      # Line 1
            TMp_x, SALA_WIDTH,      # Line 2
            SALA_WIDTH, SALA_WIDTH, # Line 3
            SALA_WIDTH, SALA_WIDTH, # Line 4
            SALA_WIDTH, SALA_WIDTH,                       # Line 5
            SALA_WIDTH, SALA_WIDTH, # Line 6
            0, 0, # Line 7
            0, 0, # Line 8
            0, 0, # Line 9
            0, 0, # Line 10
            
        ],
        y=[
            Lp_y, Lp_y,
            Rp_y, Rp_y,
            Lp_y, SALA_DEPTH / 2,
            Rp_y, SALA_DEPTH / 2,
            0, Lp_y, # Line 5
            Rp_y, SALA_DEPTH, # Line 6
            Rp_y, SALA_DEPTH, # Line 7
            0, Lp_y, # Line 8
            Lp_y, SALA_DEPTH / 2, # Line 9
            Rp_y, SALA_DEPTH / 2 # Line 10
            
        ],
        z=[
            3.3, 3.3,
            3.3, 3.3,
            3.3, 3.3,
            3.3, 3.3,
            3.3, 3.3, # Line 5
            3.3, 3.3, # Line 6
            STAGE_HEIGHT, STAGE_HEIGHT, # Line 7
            STAGE_HEIGHT, STAGE_HEIGHT, # Line 8
            STAGE_HEIGHT, STAGE_HEIGHT, # Line 9
            STAGE_HEIGHT, STAGE_HEIGHT, # Line 10
            
        ],
        mode="markers",
        marker=dict(
            size=6,
            color="black",
        ),
        name="Line Endpoints",
        showlegend=False
    )
)

# Line 5: distance from left side wall
fig.add_trace(
    go.Scatter3d(
        x=[SALA_WIDTH, SALA_WIDTH],
        y=[0, Lp_y],
        z=[3.3, 3.3],
        mode="lines",
        line=dict(
            color="blue",
            width=6,
        ),
        name=f"Distance from side = {round(Lp_y, 2)} m",
    )
)

# Line 6: distance from right side wall
fig.add_trace(
    go.Scatter3d(
        x=[SALA_WIDTH, SALA_WIDTH],
        y=[Rp_y, SALA_DEPTH],
        z=[3.3, 3.3],
        mode="lines",
        line=dict(
            color="blue",
            width=6,
        ),
        name=f"Distance from side = {round(SALA_DEPTH - Rp_y, 2)} m",
        showlegend=False
    )
)



# Line 7: distance from right side wall
fig.add_trace(
    go.Scatter3d(
        x=[0, 0],
        y=[Rp_y, SALA_DEPTH],
        z=[STAGE_HEIGHT, STAGE_HEIGHT],
        mode="lines",
        line=dict(
            color="blue",
            width=6,
        ),
        name=f"Distance from side = {round(SALA_DEPTH - Rp_y, 2)} m",
        showlegend=False
    )
)

# Line 8: distance from right side wall
fig.add_trace(
    go.Scatter3d(
        x=[0, 0],
        y=[0, Lp_y],
        z=[STAGE_HEIGHT, STAGE_HEIGHT],
        mode="lines",
        line=dict(
            color="blue",
            width=6,
        ),
        name=f"Distance from side = {round(SALA_DEPTH - Rp_y, 2)} m",
        showlegend=False
    )
)

# Line 9
fig.add_trace(
    go.Scatter3d(
        x=[0, 0],
        y=[Lp_y, SALA_DEPTH / 2],
        z=[STAGE_HEIGHT, STAGE_HEIGHT],
        mode="lines",
        line=dict(
            color="green",
            width=6,
        ),
        name=f"Distance from center = {side_shift} m",
        showlegend=False
    )
)

# Line 10
fig.add_trace(
    go.Scatter3d(
        x=[0,0],
        y=[Rp_y, SALA_DEPTH / 2],
        z=[STAGE_HEIGHT, STAGE_HEIGHT],
        mode="lines",
        line=dict(
            color="green",
            width=6,
        ),
        name=f"Distance from center = {side_shift} m",
        showlegend=False
    )
)


# =========================================================
# WINDOW
# =========================================================

W = [
    (SALA_WIDTH, SALA_DEPTH / 2 - .8 , 1.10),
    (SALA_WIDTH, SALA_DEPTH / 2 + .8, 1.10),
    (SALA_WIDTH, SALA_DEPTH / 2 + .8, 2.2),
    (SALA_WIDTH, SALA_DEPTH / 2 - .8, 2.2)   
]

w = np.array(W)

fig.add_trace(
    go.Mesh3d(
        x=w[:, 0],
        y=w[:, 1],
        z=w[:, 2],
        i=[0, 0],
        j=[1, 2],
        k=[2, 3],
        color="lightgray",
        opacity=1.0,
        flatshading=True,
        name="VENTANA",
    )
)

center = w.mean(axis=0)

fig.add_trace(
    go.Scatter3d(
        x=[center[0]],
        y=[center[1]],
        z=[center[2]],
        mode="text",
        text=["VENTANA"],
        textfont=dict(
            size=18,      # larger text
            color="black" # black text
        ),
        showlegend=False,
    )
)




# =========================================================
# FRONT RIGHT SPEAKER RECTANGLE
# =========================================================

# Front Right speaker rectangle
FR_p = [
    (0, Rp_y - speakers['Polk RC55i']['width']/2,
     STAGE_HEIGHT - speakers['Polk RC55i']['height'] - .02),

    (0, Rp_y + speakers['Polk RC55i']['width']/2,
     STAGE_HEIGHT - speakers['Polk RC55i']['height'] - .02),

    (0, Rp_y + speakers['Polk RC55i']['width']/2,
     STAGE_HEIGHT - .02),

    (0, Rp_y - speakers['Polk RC55i']['width']/2,
     STAGE_HEIGHT - .02),
]

fr = np.array(FR_p)

fig.add_trace(
    go.Mesh3d(
        x=fr[:, 0],
        y=fr[:, 1],
        z=fr[:, 2],
        i=[0, 0],
        j=[1, 2],
        k=[2, 3],
        color="black",
        opacity=1.0,
        flatshading=True,
        name="FR",
    )
)

center = fr.mean(axis=0)

fig.add_trace(
    go.Scatter3d(
        x=[center[0]],
        y=[center[1]],
        z=[center[2]],
        mode="text",
        text=["FR"],
        textfont=dict(
            size=18,      # larger text
            color="black" # black text
        ),
        showlegend=False,
    )
)

# =========================================================
# FRONT LEFT SPEAKER RECTANGLE
# =========================================================

# Front Left speaker rectangle
FL_p = [
    (0, Lp_y - speakers['Polk RC55i']['width']/2,
     STAGE_HEIGHT - speakers['Polk RC55i']['height'] - .02),

    (0, Lp_y + speakers['Polk RC55i']['width']/2,
     STAGE_HEIGHT - speakers['Polk RC55i']['height'] - .02),

    (0, Lp_y + speakers['Polk RC55i']['width']/2,
     STAGE_HEIGHT - .02),

    (0, Lp_y - speakers['Polk RC55i']['width']/2,
     STAGE_HEIGHT - .02),
]

fl = np.array(FL_p)

fig.add_trace(
    go.Mesh3d(
        x=fl[:, 0],
        y=fl[:, 1],
        z=fl[:, 2],
        i=[0, 0],
        j=[1, 2],
        k=[2, 3],
        color="black",
        opacity=1.0,
        flatshading=True,
        name="FL",
    )
)

center = fl.mean(axis=0)

fig.add_trace(
    go.Scatter3d(
        x=[center[0]],
        y=[center[1]],
        z=[center[2]],
        mode="text",
        text=["FL"],
        textfont=dict(
            size=18,      # larger text
            color="black" # black text
        ),
        showlegend=False,
    )
)

# =========================================================
# CENTER SPEAKER RECTANGLE
# =========================================================

# Center Rectangle
C_p = [(0, SALA_DEPTH / 2 - speakers['Polk 255c-RT']['width']/2, STAGE_HEIGHT - speakers['Polk 255c-RT']['height']-.02),
 (0, SALA_DEPTH / 2 + speakers['Polk 255c-RT']['width']/2, STAGE_HEIGHT - speakers['Polk 255c-RT']['height']-.02),
 (0, SALA_DEPTH / 2 + speakers['Polk 255c-RT']['width']/2, STAGE_HEIGHT -.02),
 (0, SALA_DEPTH / 2 - speakers['Polk 255c-RT']['width']/2, STAGE_HEIGHT - -.02)]


fc = np.array(C_p)

fig.add_trace(
    go.Mesh3d(
        x=fc[:, 0],
        y=fc[:, 1],
        z=fc[:, 2],
        i=[0, 0],
        j=[1, 2],
        k=[2, 3],
        color="black",
        opacity=1.0,
        flatshading=True,
        name="FC",
    )
)

center = fc.mean(axis=0)

fig.add_trace(
    go.Scatter3d(
        x=[center[0]],
        y=[center[1]],
        z=[center[2]],
        mode="text",
        text=["FC"],
        textfont=dict(
            size=18,      # larger text
            color="black" # black text
        ),
        showlegend=False,
    )
)



fig.show(renderer="browser")



In [57]:
# Save as an interactive HTML file
fig.write_html("index.html", include_plotlyjs="cdn")